# Galerie de tri des illustrations de Bibles



## 1 · Configuration

In [1]:
import os, html, json

RACINE = os.path.abspath("../../")

# Sources des illustrations de Bibles 
DOSSIER_BIBLES = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")

# Où déposer la galerie HTML
DOSSIER_GALERIE = os.path.join(RACINE, "resultats", "galerie_bibles")
os.makedirs(DOSSIER_GALERIE, exist_ok=True)

print("Illustrations  :", DOSSIER_BIBLES)
print("Galerie générée :", DOSSIER_GALERIE)

Illustrations  : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/bibles_mdz/segmentees
Galerie générée : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/galerie_bibles


## 2 · Générer la galerie

La galerie référence les images par **chemin relatif** (chargement à la demande via
`loading="lazy"`). Le fichier HTML reste léger ; les images se chargent au défilement.

In [2]:
# Lister toutes les illustrations, par Bible
bibles = sorted([d for d in os.listdir(DOSSIER_BIBLES)
                 if os.path.isdir(os.path.join(DOSSIER_BIBLES, d))])

items = []   # (bsb_id, nom_fichier, chemin_relatif depuis la galerie)
for bsb_id in bibles:
    dossier = os.path.join(DOSSIER_BIBLES, bsb_id)
    for f in sorted(os.listdir(dossier)):
        if f.lower().endswith((".jpg", ".jpeg", ".png")) and "_flip" not in f and not f.startswith("_tmp_"):
            chemin_abs = os.path.join(dossier, f)
            chemin_rel = os.path.relpath(chemin_abs, DOSSIER_GALERIE).replace(os.sep, "/")
            items.append((bsb_id, f, chemin_rel))

print(f"{len(items)} illustrations dans {len(bibles)} Bibles")

3570 illustrations dans 398 Bibles


In [3]:
# Construire le HTML
cellules = []
for bsb_id, nom, rel in items:
    ident = f"{bsb_id}/{nom}"            # identifiant unique = bsb/fichier
    cellules.append(
        f'''<div class="cell" data-id="{html.escape(ident)}" onclick="toggle(this)">
             <img loading="lazy" src="{html.escape(rel)}">
             <span class="check">&#10003;</span>
             <span class="lab">{html.escape(bsb_id)}</span>
           </div>''')

page = f'''<!DOCTYPE html><html lang="fr"><head><meta charset="utf-8">
<title>Galerie Bibles — tri</title>
<style>
  body {{ font-family: system-ui, sans-serif; margin: 0; padding: 20px; background: #faf9f7; color: #2c2c2a; }}
  h1 {{ font-size: 20px; font-weight: 500; }}
  .barre {{ position: sticky; top: 0; background: #faf9f7; padding: 12px 0; display: flex; gap: 16px;
            align-items: center; border-bottom: 1px solid #ddd; margin-bottom: 16px; z-index: 10; flex-wrap: wrap; }}
  .stat {{ font-size: 14px; }} .stat b {{ font-size: 20px; }}
  .keep {{ color: #1d9e75; }}
  button {{ font-size: 15px; padding: 8px 16px; border: 1px solid #888; background: #fff;
            border-radius: 8px; cursor: pointer; }}
  button:hover {{ background: #f0efe8; }}
  .grid {{ display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 10px; }}
  .cell {{ position: relative; border: 1px solid #ccc; border-radius: 8px; overflow: hidden;
           cursor: pointer; background: #fff; min-height: 120px; }}
  .cell img {{ width: 100%; display: block; }}
  .cell.keep {{ outline: 3px solid #1d9e75; }}
  .check {{ position: absolute; top: 4px; right: 4px; width: 26px; height: 26px; border-radius: 50%;
            background: #1d9e75; color: #fff; display: none; align-items: center; justify-content: center;
            font-size: 16px; }}
  .cell.keep .check {{ display: flex; }}
  .lab {{ position: absolute; bottom: 0; left: 0; right: 0; background: rgba(0,0,0,0.55); color: #fff;
          font-size: 10px; padding: 2px 4px; overflow: hidden; white-space: nowrap; text-overflow: ellipsis; }}
</style></head><body>
<h1>Galerie des illustrations de Bibles</h1>
<div class="barre">
  <span class="stat">À garder : <b class="keep" id="nkeep">0</b> / {len(items)}</span>
  <button onclick="exporter()">⬇ Exporter la liste à garder</button>
  <button onclick="toutGarder()">Tout garder</button>
  <button onclick="toutEffacer()">Tout décocher</button>
  <span style="font-size:13px;color:#888">Clique une image pour la GARDER (vert) · reclique pour annuler</span>
</div>
<div class="grid">{"".join(cellules)}</div>
<script>
  const total = {len(items)};
  const keep = new Set();
  function maj() {{ document.getElementById("nkeep").textContent = keep.size; }}
  function toggle(cell) {{
    const id = cell.dataset.id;
    if (keep.has(id)) {{ keep.delete(id); cell.classList.remove("keep"); }}
    else {{ keep.add(id); cell.classList.add("keep"); }}
    maj();
  }}
  function toutGarder() {{
    document.querySelectorAll(".cell").forEach(c => {{ keep.add(c.dataset.id); c.classList.add("keep"); }});
    maj();
  }}
  function toutEffacer() {{
    document.querySelectorAll(".cell").forEach(c => c.classList.remove("keep"));
    keep.clear(); maj();
  }}
  function exporter() {{
    const liste = [...keep].sort().join("\\n");
    const blob = new Blob([liste], {{type: "text/plain"}});
    const a = document.createElement("a");
    a.href = URL.createObjectURL(blob);
    a.download = "a_garder.txt";
    a.click();
  }}
</script></body></html>'''

chemin_galerie = os.path.join(DOSSIER_GALERIE, "galerie_bibles.html")
with open(chemin_galerie, "w", encoding="utf-8") as f:
    f.write(page)

print("✓ Galerie générée :", chemin_galerie)
print("\nOuvre ce fichier dans un navigateur (sur cette machine, pour que les images se chargent).")

✓ Galerie générée : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/resultats/galerie_bibles/galerie_bibles.html

Ouvre ce fichier dans un navigateur (sur cette machine, pour que les images se chargent).


In [4]:
import os
RACINE = os.path.abspath("../../")
DOSSIER_BIBLES = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")
DOSSIER_GALERIE = os.path.join(RACINE, "resultats", "galerie_bibles")

# Prendre une image au hasard et vérifier le chemin relatif
for bsb_id in sorted(os.listdir(DOSSIER_BIBLES)):
    d = os.path.join(DOSSIER_BIBLES, bsb_id)
    if not os.path.isdir(d): continue
    imgs = [f for f in os.listdir(d) if f.endswith(".jpg")]
    if imgs:
        chemin_abs = os.path.join(d, imgs[0])
        chemin_rel = os.path.relpath(chemin_abs, DOSSIER_GALERIE).replace(os.sep, "/")
        print("Image exemple   :", chemin_abs)
        print("Existe ?        :", os.path.exists(chemin_abs))
        print("Chemin relatif  :", chemin_rel)
        print("Depuis galerie  :", os.path.exists(os.path.join(DOSSIER_GALERIE, chemin_rel)))
        break

Image exemple   : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/bibles_mdz/segmentees/bsb00014585/bsb00014585_page001_det1_conf0.51.jpg
Existe ?        : True
Chemin relatif  : ../../data/bibles_mdz/segmentees/bsb00014585/bsb00014585_page001_det1_conf0.51.jpg
Depuis galerie  : True


## 3 · Appliquer le tri (après retour de Céline)



In [ ]:
import shutil

chemin_liste = os.path.join(DOSSIER_GALERIE, "a_garder.txt")   

# 1. Lire la liste des illustrations à GARDER
with open(chemin_liste, encoding="utf-8") as f:
    a_garder = set(l.strip() for l in f if l.strip())   # identifiants "bsb_id/nom.jpg"
print(f"{len(a_garder)} illustrations à garder selon Céline")

# 2. Sauvegarde de sécurité AVANT toute suppression
backup = os.path.join(RACINE, "data", "bibles_mdz", "segmentees_backup")
if not os.path.exists(backup):
    print("Sauvegarde en cours…")
    shutil.copytree(DOSSIER_BIBLES, backup)
    print(f"✓ Sauvegarde faite : {backup}")
else:
    print(f"(sauvegarde déjà existante : {backup})")

# 3. Parcourir et supprimer ce qui n'est PAS à garder
supprime, garde = 0, 0
for bsb_id in os.listdir(DOSSIER_BIBLES):
    dossier = os.path.join(DOSSIER_BIBLES, bsb_id)
    if not os.path.isdir(dossier):
        continue
    for f in os.listdir(dossier):
        if not (f.lower().endswith((".jpg", ".jpeg", ".png")) and "_flip" not in f and not f.startswith("_tmp_")):
            continue
        ident = f"{bsb_id}/{f}"
        if ident in a_garder:
            garde += 1
        else:
            os.remove(os.path.join(dossier, f))
            supprime += 1

print(f"\n✓ Terminé — {garde} gardées, {supprime} supprimées")
print(f"  (sauvegarde intacte dans {backup} si besoin de revenir en arrière)")

In [10]:
import os
RACINE = os.path.abspath("../../")
DOSSIER_BIBLES = os.path.join(RACINE, "data", "bibles_mdz", "segmentees")

total, n_bibles = 0, 0
for bsb_id in os.listdir(DOSSIER_BIBLES):
    d = os.path.join(DOSSIER_BIBLES, bsb_id)
    if not os.path.isdir(d): continue
    imgs = [f for f in os.listdir(d) if f.lower().endswith((".jpg",".jpeg",".png")) and "_flip" not in f]
    if imgs:
        n_bibles += 1
        total += len(imgs)

print(f"Après tri : {total} illustrations dans {n_bibles} Bibles")

Après tri : 731 illustrations dans 131 Bibles


In [9]:
for bsb_id in vides:
    d = os.path.join(DOSSIER_BIBLES, bsb_id)
    shutil.rmtree(d)
    print(f"  supprimé : {bsb_id}")

print(f"\n✓ {len(vides)} dossiers vides supprimés")

# Vérification finale
restants = [d for d in os.listdir(DOSSIER_BIBLES)
            if os.path.isdir(os.path.join(DOSSIER_BIBLES, d))]
print(f"Il reste {len(restants)} dossiers de Bibles (avec illustrations)")

  supprimé : bsb00029452
  supprimé : bsb00050923
  supprimé : bsb00050999
  supprimé : bsb00053608
  supprimé : bsb00065386
  supprimé : bsb00065395
  supprimé : bsb00066020
  supprimé : bsb00085609
  supprimé : bsb00085641
  supprimé : bsb00085772
  supprimé : bsb00127878
  supprimé : bsb10030243
  supprimé : bsb10086304
  supprimé : bsb10086305
  supprimé : bsb10141273
  supprimé : bsb10141274
  supprimé : bsb10141275
  supprimé : bsb10141276
  supprimé : bsb10141277
  supprimé : bsb10141278
  supprimé : bsb10141292
  supprimé : bsb10152177
  supprimé : bsb10152187
  supprimé : bsb10173752
  supprimé : bsb10173759
  supprimé : bsb10173763
  supprimé : bsb10173765
  supprimé : bsb10173766
  supprimé : bsb10173767
  supprimé : bsb10173769
  supprimé : bsb10173770
  supprimé : bsb10173775
  supprimé : bsb10173777
  supprimé : bsb10174468
  supprimé : bsb10196330
  supprimé : bsb10196334
  supprimé : bsb10196346
  supprimé : bsb10199027
  supprimé : bsb10199033
  supprimé : bsb10199034
